# H1L1 — RoPE on a small example

**Week 6, notebook 1.** Parts 1 and 2 go with the end of Hour 1, parts 3 and 4 with Hour 2. The slides state the property of RoPE in one line,

$$\langle R_m q,\; R_n k\rangle \;=\; \langle q,\; R_{n-m} k\rangle ,$$

so the score depends only on the offset $n - m$. This notebook works through
that line on the example of the slides, a head with one pair ($d_{head} = 2$,
$\theta = 1$), and then on the 64 pairs of a real head. It has four parts:

1. **What rotates.** A pair is two numbers read as one point in a plane; $j$
   sets the rate and the position $m$ sets the angle.
2. **What the score depends on.** Only the lengths and the difference of the
   angles, so a common rotation cancels and only the offset remains.
3. **Six keys and one query.** The angle of the query selects a distance.
4. **What this gives.** The origin cancels, which the KV cache relies on, and
   the slow pairs never complete a rotation, which YaRN relies on.

**Colab:** Runtime → Run all. A few seconds, NumPy only.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
C_BLUE, C_RED, C_TEAL, C_PURPLE, C_GRAY = \
    "#1F4E79", "#C00000", "#0E7C7B", "#7030A0", "#595959"

# ---------------------------------------------------------------- KNOB
L_TRAIN = 4096      # training length used in the spectrum section
# ----------------------------------------------------------------------

def rotate(pair, angle):
    c, s = math.cos(angle), math.sin(angle)
    return np.array([pair[0] * c - pair[1] * s, pair[0] * s + pair[1] * c])

def circle_ax(ax, r=1.35):
    t = np.linspace(0, 2 * np.pi, 200)
    ax.plot(np.cos(t), np.sin(t), color=C_GRAY, lw=0.6, alpha=0.5)
    ax.axhline(0, color=C_GRAY, lw=0.4, alpha=0.4)
    ax.axvline(0, color=C_GRAY, lw=0.4, alpha=0.4)
    ax.set_xlim(-r, r), ax.set_ylim(-r, r), ax.set_aspect("equal")
    ax.set_xticks([]), ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

def arrow(ax, v, color, label=None, lw=2.2, alpha=1.0):
    ax.annotate("", xy=v, xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=color, lw=lw, alpha=alpha))
    if label:
        ax.annotate(label, xy=v * 1.16, ha="center", va="center", fontsize=9,
                    color=color, alpha=alpha)


# Part 1 of 4 — What rotates

### A pair is two adjacent numbers, read as one point

One head's query with $d_{head}=8$ is not eight numbers — it is **four
points**, one per plane. `q[2j]` and `q[2j+1]` are the $x$ and $y$ of point
$j$. Rotating moves each point around **its own** origin; pair 0 never
meets pair 1.


In [ ]:
q8 = np.array([0.90, 0.40, 0.30, -0.70, 1.10, 0.20, -0.50, 0.60])
print("the vector:  ", "  ".join(f"{v:+.2f}" for v in q8))
print("read as pairs:", "  ".join(f"[{q8[2*j]:+.2f},{q8[2*j+1]:+.2f}]"
                                  for j in range(4)))

fig, axes = plt.subplots(1, 4, figsize=(11, 3.0))
for j, ax in enumerate(axes):
    circle_ax(ax)
    arrow(ax, q8[2 * j:2 * j + 2], C_BLUE)
    ax.set_title(f"pair {j} = (q[{2*j}], q[{2*j+1}])", fontsize=9)
fig.suptitle("one 8-number query = four arrows on four separate planes", fontsize=11)
plt.tight_layout(); plt.show()


### $j$ sets the rate, $m$ sets the amount

Each pair $j$ gets its own rotation rate $\theta_j = 10000^{-2j/d}$ —
pair 0 fast, the last pair slow. A token at sentence position $m$ has every
pair turned by $m\,\theta_j$.

Below: the **same word** (same eight numbers out of $W_Q$) placed at
positions $m = 0,1,2,3$. Fast pairs fan out; the slow pair barely moves.
The fan is the position code.


In [ ]:
theta8 = 10000.0 ** (-2.0 * np.arange(4) / 8)
print("rates:", "  ".join(f"theta_{j}={t:.4f} rad/pos" for j, t in enumerate(theta8)))

fig, axes = plt.subplots(1, 4, figsize=(11, 3.2))
colors_m = plt.cm.viridis(np.linspace(0.05, 0.85, 4))
for j, ax in enumerate(axes):
    circle_ax(ax)
    for m in range(4):
        arrow(ax, rotate(q8[2 * j:2 * j + 2], m * theta8[j]), colors_m[m],
              label=f"m={m}" if j == 0 else None, lw=1.8,
              alpha=0.45 + 0.55 * (m == 3))
    ax.set_title(f"pair {j}:  turns {theta8[j]:.3f} rad per position", fontsize=9)
fig.suptitle("one word at four positions: j picks the speed, m picks how far",
             fontsize=11)
plt.tight_layout(); plt.show()

lengths = [np.linalg.norm(rotate(q8[0:2], m * theta8[0])) for m in range(4)]
print("pair 0 length at m=0..3:", " ".join(f"{v:.4f}" for v in lengths),
      " -- rotation is rigid, length never moves")


# Part 2 of 4 — What the score depends on

The dot product of two pairs reads exactly three things: both lengths and
the **angle between them** — $q\cdot k = r_q r_k \cos(\varphi_k -
\varphi_q)$. Each angle *alone* is invisible to it. RoPE hides position in
that blind spot: rotating **both** arrows by a shared amount cancels in the
difference, and what does not cancel is $ (n - m)\,\theta$ — the offset.

Measured, every pair of positions with the same offset:


In [ ]:
k8 = np.array([0.30, 0.90, -0.40, 0.50, 0.70, -0.10, 0.20, 0.80])

def rope8(v, m):
    return np.concatenate([rotate(v[2 * j:2 * j + 2], m * theta8[j])
                           for j in range(4)])

print(f"{'q at m':>7} {'k at n':>7} {'offset':>7} {'score q·k':>12}")
for m, n in [(0, 0), (3, 3), (900, 900), (0, 2), (3, 5), (100, 102), (900, 902)]:
    s = rope8(q8, m) @ rope8(k8, n)
    print(f"{m:>7} {n:>7} {n - m:>7} {s:>12.6f}")
print("-> offset 0 gives one score, offset 2 another; the absolute slots cancel.")


# Part 3 of 4 — Six keys and one query

### Five keys, one query: watching an arrow find its target

Now the slides' smallest possible head: $d_{head} = 2$ (ONE pair),
$\theta = 1$ rad/position, sentence **"the capital of france is ___"**.

All five keys start as the same arrow $[1, 0]$ — identical content, so
*nothing but position* can distinguish them. RoPE spins each key by
$n\cdot 1$ radian. The query sits in slot 5 — the position about to
produce **"paris"** — and its content points it at $\varphi_q = -2$:
**two radians behind**.

Both arrows spin forward at the same rate, and the query — sitting two
positions later than "france" — accumulates exactly two radians more. So
it lands **on top of** "france": identical numbers, dot product exactly 1.


In [ ]:
TOK = "the capital of france is".split()          # 5 keys, n = 0..4
TH, M_Q, PHI_Q = 1.0, 5, -2.0            # the query sits in slot 5
q_raw = np.array([math.cos(PHI_Q), math.sin(PHI_Q)])
q_rot = rotate(q_raw, M_Q * TH)
keys_rot = [rotate(np.array([1.0, 0.0]), n * TH) for n in range(len(TOK))]
scores = np.array([q_rot @ k for k in keys_rot])

fig = plt.figure(figsize=(11.5, 4.6))
ax = fig.add_subplot(1, 2, 1)
circle_ax(ax, r=1.5)
for n, k in enumerate(keys_rot):
    hit = abs(scores[n] - 1) < 1e-9
    arrow(ax, k, C_RED if hit else C_GRAY, f"{TOK[n]} (n={n})",
          lw=2.6 if hit else 1.4, alpha=1.0 if hit else 0.55)
arrow(ax, q_rot * 0.85, C_BLUE, None, lw=3.0)
ax.annotate('query: slot m=5,\nabout to say "paris"', xy=q_rot * 0.5,
            xytext=(-1.45, -1.25), fontsize=9, color=C_BLUE)
ax.set_title("five keys, spun by their positions — the query lands on ONE of them",
             fontsize=10)

ax = fig.add_subplot(1, 2, 2)
bars = ax.bar(range(len(TOK)), scores, color=[C_RED if abs(s - 1) < 1e-9 else C_GRAY
                                       for s in scores])
for n, s in enumerate(scores):
    ax.text(n, s + (0.05 if s >= 0 else -0.12), f"{s:+.3f}", ha="center", fontsize=9)
ax.set_xticks(range(len(TOK)))
ax.set_xticklabels([f"{t}\nn={n}" for n, t in enumerate(TOK)])
ax.axhline(0, color=C_GRAY, lw=0.6)
ax.set_ylim(-1.3, 1.35), ax.set_ylabel("q · k")
ax.set_title(f"scores: the key {int(-PHI_Q/TH)} back is an exact match", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

print("The query's CONTENT angle (-2 rad) is a DISTANCE REQUEST: 'the token 2 back'")
print('  -> which, in this sentence, is the word "france".')
print("phi_q + m*theta = phi_k + n*theta  =>  phi_q = phi_k - D*theta.")


### And what the query's angle means: content picks the curve

So does position *overwrite* the query? No — it **adds**. In one pair the
score is

$$\text{score} = r_q r_k \cos\big(\underbrace{(\varphi_k - \varphi_q)}_{\text{content}} - \underbrace{D\,\theta}_{\text{position}}\big)$$

Change the content angle $\varphi_q$ and the *same* cosine curve slides
sideways: the peak moves to a different distance $D$. **Content decides
which distance to ask for; position decides which column of the curve you
read.** Neither erases the other.


In [ ]:
phis = [0.0, -1.0, -2.0, -3.0]
Ds = np.arange(0, 7)
table = np.array([[math.cos(-p - D * TH) for D in Ds] for p in phis])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 3.6),
                             gridspec_kw={"width_ratios": [1.15, 1]})
im = a1.imshow(table, cmap="RdBu_r", vmin=-1, vmax=1)
for i in range(len(phis)):
    for j in range(len(Ds)):
        hit = table[i, j] > 0.99
        a1.text(j, i, f"{table[i, j]:+.2f}", ha="center", va="center",
                fontsize=8.5, fontweight="bold" if hit else "normal",
                color="white" if abs(table[i, j]) > 0.6 else "#222222")
a1.set_xticks(Ds), a1.set_xlabel("distance back D")
a1.set_yticks(range(len(phis)))
a1.set_yticklabels([f"$\\varphi_q$={p:+.0f}" for p in phis])
a1.set_title("the same numbers, one column further each row", fontsize=10)

Dc = np.linspace(0, 7, 300)
for p, c in zip(phis, [C_RED, C_TEAL, C_BLUE, C_PURPLE]):
    a2.plot(Dc, np.cos(-p - Dc * TH), color=c, lw=1.5,
            label=f"$\\varphi_q$={p:+.0f} → peak at D={-p:.0f}")
    a2.plot(Ds, [math.cos(-p - D * TH) for D in Ds], "o", color=c, ms=4)
a2.set_xlabel("distance back D"), a2.set_ylabel("score")
a2.set_title("one curve, slid; evaluated only at integer D (dots)", fontsize=10)
a2.legend(fontsize=7.5, frameon=False, loc="lower right")
a2.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()


# Part 4 of 4 — What this gives: the KV cache and context extension

### The origin cancels — which is why the KV cache works

Every rotation above used the **absolute** position — $1004\,\theta$ is a
big angle. Shift the whole sentence a thousand positions to the right and
every score is *bit-identical* (in float64: exactly), because whatever is
added to both $m$ and $n$ subtracts straight back out.

This is the property that the KV cache relies on: **rotate each key
once, by its absolute position, at insertion — and never touch it again.**


In [ ]:
print(f"{'D':>3} {'at pos 5-and-back':>19} {'at pos 1005-and-back':>21} {'difference':>12}")
for D in range(1, 6):
    outs = []
    for off in (0, 1000):
        qa = rotate(q_raw, (off + M_Q) * TH)
        ka = rotate(np.array([1.0, 0.0]), (off + M_Q - D) * TH)
        outs.append(qa @ ka)
    print(f"{D:>3} {outs[0]:>19.12f} {outs[1]:>21.12f} {abs(outs[0]-outs[1]):>12.1e}")


### The spectrum — and the pairs that never learned to be relative

A real head has $d_{head}=128$: **64 pairs**, rates spread geometrically
from $\theta_0 = 1$ (a full turn every ~6 tokens) to $\theta_{63} \approx
10^{-4}$ (a full turn every ~55,000 tokens). Fast pairs give sharp local
resolution; slow pairs make sure no two distances in the window ever look
alike. It is a positional number system — the reason a clock has three
hands instead of one very precise one.

And the fact that context extension relies on: at $L_{train} = 4096$, the
slowest pairs **never completed a single turn during training**. Their
score was a monotone ramp over everything the model ever saw — an absolute
position code wearing a relative one's clothes. Those are exactly the
pairs naive extension breaks, and exactly the ones YaRN treats specially
(deck §2.8).


In [ ]:
D_HEAD = 128
j = np.arange(D_HEAD // 2)
theta = 10000.0 ** (-2.0 * j / D_HEAD)
wavelength = 2 * np.pi / theta
never = int((wavelength > L_TRAIN).sum())
first = int(np.argmax(wavelength > L_TRAIN))
print(f"L_train = {L_TRAIN}: {never} of 64 pairs never completed one turn "
      f"(pairs j >= {first})")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.2),
                             gridspec_kw={"width_ratios": [1, 1.35]})
a1.semilogy(j, wavelength, color=C_GRAY, lw=1.2)
a1.scatter(j[:first], wavelength[:first], s=14, color=C_GRAY, zorder=3)
a1.scatter(j[first:], wavelength[first:], s=14, color=C_PURPLE, zorder=3)
a1.axhline(L_TRAIN, color=C_PURPLE, ls="--", lw=1.1)
a1.text(2, L_TRAIN * 1.3, f"$L_{{train}}$ = {L_TRAIN:,}", fontsize=9, color=C_PURPLE)
a1.annotate(f"{never} of 64 pairs:\nsecretly absolute",
            xy=(first + 3, wavelength[first + 3]), xytext=(20, 2e5),
            fontsize=9, color=C_PURPLE,
            arrowprops=dict(arrowstyle="->", color=C_PURPLE, lw=1))
a1.set_xlabel("pair index j"), a1.set_ylabel("wavelength (tokens, log)")
a1.set_title("(a) each pair's wavelength", fontsize=10, loc="left")
a1.spines[["top", "right"]].set_visible(False)

DW = 1024
grid = np.cos(np.arange(DW)[None, :] * theta[:, None])
im = a2.imshow(grid, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1,
               interpolation="nearest", extent=(0, DW, 63.5, -0.5))
a2.axhline(first - 0.5, color="black", ls="--", lw=1.2)
a2.text(DW * 0.03, first + 4, "below the line: a smooth gradient,\nnot a stripe "
        "pattern — these pairs\nnever learned to repeat", fontsize=8, va="top",
        bbox=dict(fc="white", ec="none", alpha=0.85, pad=2))
a2.set_xlabel(f"offset D (first {DW:,} tokens)"), a2.set_ylabel("pair index j")
a2.set_title("(b) every pair's score vs offset — the position code, whole",
             fontsize=10, loc="left")
plt.colorbar(im, ax=a2, fraction=0.046, label=r"$\cos(D\,\theta_j)$")
plt.tight_layout(); plt.show()


## What H1L1 showed

| claim | the number that settles it |
|---|---|
| a pair is one point on a plane | 8 numbers = 4 arrows; lengths never move under rotation |
| $j$ = rate, $m$ = amount | the fan in Part 2: same word, four positions |
| only the offset survives | same-offset scores identical at positions 0/3/900 |
| the query's angle is a distance request | $\varphi_q = -1$ landed exactly on the key 1 back |
| content adds to position, never erased | one cosine curve, slid one column per $\varphi_q$ |
| the origin cancels | scores at 5 vs 1005: difference 0.0 → the KV cache |
| some pairs are secretly absolute | at $L_{train}$=4096: the slow pairs never finished a turn |

**Next (H2L1):** the angles are no longer set by hand; gradient descent
chooses them.
